# Evaluate bo767 retrieval recall accuracy with Docling and Milvus

In this notebook, we'll get the end-to-end recall accuracy of a retrieval pipeline made up of Docling's extraction and embedding tasks and a Milvus vector database (VDB).

Refer to the [Download Bo20 and Bo767](digital_corpora_download.ipynb) notebook to fetch the **Bo767** dataset consisting of 767 PDF documents. 

## Settings

### Environment and dependencies

Create and activate a dedicated environment:

```shell
uv venv --python 3.12 .venv && source .venv/bin/activate && uv pip install ipykernel --upgrade
```

Install the depdencies:

In [3]:
import os
import logging
logging.getLogger().setLevel(logging.WARNING)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

! uv pip install -q --upgrade docling ipython ipywidgets pymilvus openai

### Installing Milvus

You need either [Milvus Lite](https://milvus.io/docs/milvus_lite.md) or Milvus Standalone (for larger data scale). This notebook assumes that you have a Milvus Standalone instance. 

You can install it on Docker following the instructions on [Milvus in Docker](https://milvus.io/docs/install_standalone-docker.md) or use Podman as an alternative:

- Create a directory to run the installation script and store Milvus data. You will also need to create a directory for the data:

  ```bash
  mkdir -p milvus/volumes/milvus
  cd milvus
  ```
- Download the installation script:

  ```bash
  curl -sfL https://raw.githubusercontent.com/milvus-io/milvus/master/scripts/standalone_embed.sh -o standalone_embed.sh
  ```

- Modify the script. On macOS, simply type:

  ```bash
  sed -i '' 's/docker/podman/g' standalone_embed.sh
  ```

- Start the container:

  ```bash
  bash standalone_embed.sh start
  ```

- Some useful commands to upgrade, stop, and delete the Milvus container:

  ```bash
  bash standalone_embed.sh upgrade
  bash standalone_embed.sh stop
  bash standalone_embed.sh delete
  ```

## Convertion & Ingestion

In [4]:
# Fetch the files
from pathlib import Path

pdf_files = list(Path("../data/bo767").rglob("*.pdf"))

In [ ]:
# Convert with Docling (standard pipeline) and save as JSON

import os
import time
from pathlib import Path

from docling.datamodel.base_models import ConversionStatus
from docling.document_converter import DocumentConverter

# Optional: save documents in JSON format
os.makedirs("docs/bo767", exist_ok=True)
pdf_files = [item for item in pdf_files if not os.path.isfile(
    Path("docs/bo767") / item.with_suffix(".json").name)]
converter = DocumentConverter()
start_time = time.time()
conv_results = converter.convert_all(pdf_files, raises_on_error=False)
success: int = 0
for item in conv_results:
    if item.status == ConversionStatus.SUCCESS:
        item.document.save_as_json((
            Path("docs/bo767") / item.document.origin.filename).with_suffix(".json"))
        success += 1
end_time = time.time() - start_time

print(f"Converted {success} out of {len(pdf_files)} files in {end_time:.2f} seconds.")

In [ ]:
# Set up the embeddings with OpenAI and Nvidia API
# Hybrid chunker will ensure to produce chunks below maximum token length
# Page chunker may produce higher length: discard the end 
from typing import Literal
from openai import OpenAI

client = OpenAI(
  api_key="nvapi-xxxxx",
  base_url="https://integrate.api.nvidia.com/v1"
)

def emb_text(text: str, input_type: Literal["passage", "query"]):
    return (
        client.embeddings.create(
            model="nvidia/llama-3.2-nv-embedqa-1b-v2",
            encoding_format="float",
            extra_body={"input_type": input_type, "truncate": "END"},
            input=text,
        ).data[0].embedding
    )

test_embedding = emb_text("This is a test", input_type="passage")
embedding_dim = len(test_embedding)
print(f"Embedding size is {embedding_dim}")

Embedding size is 2048


In [6]:
# Set up Docling's chunkers: page chunker and hybrid chunker using llama 3.2 tokenizer
from transformers import AutoTokenizer
from docling.chunking import HybridChunker
from docling_core.transforms.chunker import PageChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

page_chunker = PageChunker()

from_pretrained = AutoTokenizer.from_pretrained("nvidia/llama-nemotron-embed-1b-v2")
# max tokens needs to be set explicitly: 8192 tokens accoring to model card
tokenizer = HuggingFaceTokenizer(
    tokenizer=from_pretrained,
    max_tokens = 8192
)
hybrid_chunker = HybridChunker(tokenizer=tokenizer)

In [15]:
# Chunk iterator
import statistics
from typing import Iterator
from docling_core.types import DoclingDocument
from docling_core.transforms.chunker import BaseChunker, DocChunk

def get_chunks(chunker: BaseChunker) -> Iterator[list[DocChunk]]:
    json_files = Path("docs/bo767").rglob("*.json")
    doc_num: int = 0
    chunk_num: list[int] = []
    for item in json_files:
        with open(item, encoding="utf-8") as file_handler:
            json_doc = file_handler.read()
        doc = DoclingDocument.model_validate_json(json_doc)
        doc_chunks = list(chunker.chunk(doc))
        chunk_num.append(len(doc_chunks))
        doc_num += 1
        yield doc_chunks

    print(f"Chunks in {doc_num} docs: {sum(chunk_num)=}, {statistics.mean(chunk_num)=}, {statistics.median(chunk_num)=}, {max(chunk_num)=}, {min(chunk_num)=}")

In [ ]:
# Set ingestion: Milvus Standalone
# Only dense embeddings (to be aligned with bo767_recall.ipynb)
# taking milvus index parameters from implicit values in NV Ingest client
from pymilvus import MilvusClient, DataType

CONSISTENCY = "Bounded"
INDEX_TYPE = "FLAT"
METRIC_TYPE = "L2"

milvus_client = MilvusClient(uri="http://localhost:19530")
collection_base = "bo767_docling"
collections = {"page", "section", "contextualized"}
for item in collections:
    collection_name = f"{collection_base}_{item}"
    if milvus_client.has_collection(collection_name):
        milvus_client.drop_collection(collection_name)

    schema = MilvusClient.create_schema(
        auto_id=False,
        enable_dynamic_field=False,
    )
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=embedding_dim)
    schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)
    schema.add_field(field_name="filename", datatype=DataType.VARCHAR, max_length=20)
    schema.add_field(field_name="page_no", datatype=DataType.INT16)

    index_params = milvus_client.prepare_index_params()
    index_params.add_index(field_name="id", index_type="STL_SORT")
    index_params.add_index(field_name="vector", index_type=INDEX_TYPE, metric_type=METRIC_TYPE)

    milvus_client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params,
        consistency_level=CONSISTENCY,
    )

In [9]:
# Supporting function to avoid exceeding max length
def truncate_utf8(text, max_bytes):
    data = text.encode("utf-8")
    if len(data) <= max_bytes:
        return text
    
    truncated = data[:max_bytes]

    return truncated.decode("utf-8", errors="ignore")

In [ ]:
# Insert data with page chunker
# Docling assumption: the first provenance of the first doc item holds the 'right' page number
from tqdm import tqdm

logging.getLogger().setLevel(logging.WARNING)
total = len(list(Path("docs/bo767").rglob("*.json")))
collection_name = f"{collection_base}_page"
idx = 0
data = []
for chunks in tqdm(get_chunks(page_chunker), total=total, desc="Embedding page chunks"):
    data.clear()
    for chunk in chunks:
        embedding = emb_text(chunk.text, "passage")
        data.append(
            {"id": idx,
             "vector": embedding,
             "text": truncate_utf8(chunk.text, 65535),
             "filename": chunk.meta.origin.filename,
             "page_no": chunk.meta.doc_items[0].prov[0].page_no,
            }
        )
        idx += 1
    milvus_client.insert(collection_name=collection_name, data=data)

Embedding page chunks: 100%|██████████| 767/767 [1:22:12<00:00,  6.43s/it]   

Chunks in 767 docs: sum(chunk_num)=9171, statistics.mean(chunk_num)=70.00763358778626, statistics.median(chunk_num)=20, max(chunk_num)=1443, min(chunk_num)=0


In [ ]:
# Insert data with hybrid chunker
from tqdm import tqdm

logging.getLogger().setLevel(logging.WARNING)
total = len(list(Path("docs/bo767").rglob("*.json")))
hybrid_collections = {f"{collection_base}_section", f"{collection_base}_contextualized"}
hybrid_data = [[], []]
num = 0
for chunks in tqdm(get_chunks(hybrid_chunker), total=total, desc="Embedding section chunks"):
    for data in hybrid_data:
        data.clear()
    for chunk in chunks:
        for idx, collection_name in enumerate(hybrid_collections):
            text = hybrid_chunker.contextualize(chunk) if collection_name.endswith("contextualized") else chunk.text
            embedding = emb_text(text, "passage")
            hybrid_data[idx].append(
                {"id": num,
                "vector": embedding,
                "text": truncate_utf8(text, 65535),
                "filename": chunk.meta.origin.filename,
                "page_no": chunk.meta.doc_items[0].prov[0].page_no,
                }
            )
        num += 1
    for idx, collection_name in enumerate(hybrid_collections):
        milvus_client.insert(collection_name=collection_name, data=hybrid_data[idx])

Embedding section chunks:   5%|▌         | 39/767 [13:55<6:38:09, 32.82s/it] 

## Evaluation

In [20]:
# Supporting function to calculate the evaluation metrics: Recal@K
import numpy as np
import pandas as pd
from collections import defaultdict
from pymilvus.client.types import LoadState
from typing import Any

def get_recall_scores(query_df: pd.DataFrame, query_embeddings: list[Any], collection_name: str):
    if milvus_client.get_load_state(collection_name)["state"] == LoadState.NotLoad:
        milvus_client.load_collection(collection_name)
    else:
        milvus_client.refresh_load(collection_name)

    hits = defaultdict(list)    
    all_answers = milvus_client.search(
        collection_name=collection_name,
        data=query_embeddings,
        limit=10,
        consistency_level=CONSISTENCY,
        search_params={"ef": 100},
        output_fields=["filename", "page_no"],
    )

    for idx in range(len(query_df)):
        expected_pdf_page = query_df["pdf_page"][idx]
        retrieved_answers = all_answers[idx]
        retrieved_pdfs = [result["entity"]["filename"].removesuffix(".pdf") for result in retrieved_answers]
        retrieved_pages = [result["entity"]["page_no"] - 1 for result in retrieved_answers]
        retrieved_pdf_pages = [f"{pdf}_{page}" for pdf, page in zip(retrieved_pdfs, retrieved_pages)]

        for k in [1, 3, 5, 10]:
            hits[k].append(expected_pdf_page in retrieved_pdf_pages[:k])
    
    for k in hits:
        print(f"  - Recall @{k}: {np.mean(hits[k]) :.3f}")

### Text Recall

In [16]:
import pandas as pd

df_query = pd.read_csv('../data/text_query_answer_gt_page.csv')
df_query['pdf_page'] = df_query.apply(lambda x: f"{x.pdf.removesuffix('.pdf')}_{x.gt_page}", axis=1) 
df_query

,pdf,query,answer,gt_page,pdf_page
0,1102434.pdf,How much was the ARtillery Intelligence projec...,$4.2 billion,19,1102434_19
1,1102434.pdf,How much revenue of AR advertising is expected...,$8.8 billion,3,1102434_3
2,1096078.pdf,What types of statistics were utilized by Rein...,descriptive statistics,3,1096078_3
3,1054125.pdf,What was the maximum amount requested for cond...,"$35,000.00",1,1054125_1
4,1246906.pdf,What is the median household income for the Ci...,"$53,278",7,1246906_7
...,...,...,...,...,...
483,2089825.pdf,Under the Climate Action and Low Carbon Develo...,Denis Naughten TD,0,2089825_0
484,2089825.pdf,How many organizations make up Stop Climate Ch...,30,5,2089825_5
485,2098077.pdf,What is the maximum length of Sai Yok bent-­to...,2.4 inches,1,2098077_1
486,2098077.pdf,What characteristic sets the Sai Yok Bent-toed...,enlarged thigh scales,1,2098077_1


In [22]:
# Calculate the query embeddings
query_embeddings = []
for question in tqdm(df_query["query"].to_list(), desc="Calculating the query embeddings"):
    query_embeddings.append(emb_text(question, "query"))

Calculating the query embeddings: 100%|██████████| 488/488 [02:34<00:00,  3.16it/s]


In [28]:
for item in collections:
    collection_name = f"{collection_base}_{item}"
    print(f"Evaluation of {collection_name} on Text")
    get_recall_scores(df_query, query_embeddings, collection_name)

Evaluation of bo767_docling_page on Text
  - Recall @1: 0.627
  - Recall @3: 0.846
  - Recall @5: 0.889
  - Recall @10: 0.928
Evaluation of bo767_docling_section on Text
  - Recall @1: 0.596
  - Recall @3: 0.707
  - Recall @5: 0.746
  - Recall @10: 0.787
